# 06 - Weekly Forecast Workflow ("Deployment")

**What "deploying" means for a project this size:** not a live server or an app — just a **repeatable routine**. Once a week, you (1) add the past week's actual covers to the history file, (2) fill in what you already know about next week (calendar, weather forecast, planned promos, expected reservations) in a small spreadsheet, and (3) run this notebook top to bottom to get a forecast for the next 7 days. No step requires touching the modeling code itself.

**Why exactly 7 days ahead, not further:** two of our features — `covers_lag_7` and `covers_rollavg_7d_trend` — need real, already-happened covers from up to 7 days before the date being forecast. If you tried to forecast 10 days out, day 10's `covers_lag_7` would fall on day 3 — which is *itself* a forecast, not a real number yet, and using a forecast to build a forecast compounds error fast. Capping the horizon at 7 days guarantees every input is a real, already-known value. Re-running this notebook weekly naturally rolls the window forward.

## Step 1: Refresh the historical data

Before running the rest of this notebook, make sure `data/processed/covers_daily_clean.csv` includes last week's actual covers (and reservations, weather, any holiday/event/promo flags) — export from the POS/reservations system the same way notebook 01 did, and append the new rows. This notebook always retrains on *everything* in that file, so keeping it current is what keeps the forecast current — there's no separate "retraining" step to remember.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

full = pd.read_csv("../data/processed/covers_daily_clean.csv", parse_dates=["date"])
full = full.sort_values("date").reset_index(drop=True)
covers_by_date = full.set_index("date")["covers"]

hist = full[full["is_closed"] == 0].reset_index(drop=True).copy()
print(f"History runs through {hist['date'].max().date()} — {len(hist)} open days on record")

## Step 2: Add reservations as a feature when available, and refit on all history

**A feature notebook 05 didn't use, but should when it exists:** `reservations_covers` — the final reservations count for a date — correlates with total covers at **0.94** in this restaurant's data, by far the strongest single signal available. That makes sense: reservations are often close to half of a night's covers directly, and a heavily-booked night usually pulls in more walk-ins too.

**Not every restaurant tracks this, though** — plenty only keep a covers backlog with no reservations breakdown. So this notebook checks whether the column is actually present and mostly filled in before using it: if yes, it's added as a feature (and the input sheet's `expected_reservations` column matters); if no, the model just falls back to the notebook 05 feature set (lags, rolling averages, day-of-week, holiday/event/promo, weather) with no changes needed on your part.

**If you do have it, one important caveat:** the model is trained on *final* day-of reservation totals. When you fill in next week's input sheet, `expected_reservations` should be your **best estimate of where reservations will land by service time** — informed by current bookings and how that weekday usually paces — not just today's partial booking count. Entering a partial count as if it were final will bias the forecast low, especially for dates further out where fewer of the eventual reservations have come in yet.

We also drop the train/test split here — for a deployed forecast you want the model trained on every day of history you have, not held-back test data. (Notebook 05 already answered "does this modeling approach work" with that held-out test; here we're using the same approach in production mode.)

In [ ]:
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

def get_lag(target_date, days_back):
    """Actual covers exactly `days_back` calendar days before target_date (NaN if unknown)."""
    return covers_by_date.get(target_date - pd.Timedelta(days=days_back), np.nan)

def get_rollavg_weekday(target_date, dow, history, n=4):
    """Average covers from the n most recent same-weekday dates strictly before target_date."""
    past = history[(history["date"] < target_date) & (history["day_of_week"] == dow)].sort_values("date").tail(n)
    return past["covers"].mean() if len(past) else np.nan

def get_rollavg_7d_trend(target_date, lag=7, window=7, min_periods=3):
    """Average covers over a 7-day window ending `lag` days before target_date."""
    window_dates = [target_date - pd.Timedelta(days=i) for i in range(lag, lag + window)]
    vals = [covers_by_date.get(d) for d in window_dates]
    vals = [v for v in vals if pd.notna(v)]
    return np.mean(vals) if len(vals) >= min_periods else np.nan

# These are the same three feature ideas from notebook 04, rewritten as reusable functions
# so Step 4 can apply the *identical* logic to future dates, not just historical ones.

hist["day_of_week"] = pd.Categorical(hist["day_of_week"], categories=dow_order)
hist["covers_lag_7"] = hist["date"].apply(lambda d: get_lag(d, 7))
hist["covers_lag_14"] = hist["date"].apply(lambda d: get_lag(d, 14))
hist["covers_rollavg_4wk_weekday"] = [get_rollavg_weekday(r.date, r.day_of_week, hist) for r in hist.itertuples()]
hist["covers_rollavg_7d_trend"] = hist["date"].apply(get_rollavg_7d_trend)
hist["is_rain"] = (hist["precip_in"] > 0).astype(int)

if "reservations_covers" in hist.columns:
    corr = hist["covers"].corr(hist["reservations_covers"])
    print(f"Correlation of reservations_covers with covers: {corr:.2f}" if pd.notna(corr) else "reservations_covers column present but not enough data to correlate")
else:
    print("No reservations_covers column in this dataset.")

In [ ]:
has_reservations = (
    "reservations_covers" in hist.columns
    and hist["reservations_covers"].notna().mean() >= 0.8  # at least 80% of days have it logged
)

numeric_feats = [
    "covers_lag_7", "covers_lag_14",
    "covers_rollavg_4wk_weekday", "covers_rollavg_7d_trend",
    "is_holiday", "is_local_event", "is_promo",
    "temp_high_f", "is_rain",
]
if has_reservations:
    numeric_feats.append("reservations_covers")
    print(f"reservations_covers found ({hist['reservations_covers'].notna().mean():.0%} of days logged) — using it as a feature.")
else:
    print("No usable reservations_covers column found — training on covers history + calendar/weather features only.")

dow_dummies = pd.get_dummies(hist["day_of_week"], prefix="dow", drop_first=True).astype(float)
X_hist = pd.concat([dow_dummies, hist[numeric_feats]], axis=1)
y_hist = hist["covers"]

complete = X_hist.notna().all(axis=1) & y_hist.notna()
X_train, y_train = X_hist[complete], y_hist[complete]
print(f"Training on {len(X_train)} of {len(hist)} historical days (rest missing an early lag or a flagged missing target)")

X_design = np.column_stack([np.ones(len(X_train)), X_train.values])
coefs, _, rank, _ = np.linalg.lstsq(X_design, y_train.values, rcond=None)
print(f"Fit {len(coefs)} coefficients, matrix rank {rank} (full rank = no exact redundancy)")

## Step 3: Load next week's inputs

Reads the `Upcoming_Week` tab of `upcoming_week_template.xlsx` — the sheet you fill in by hand each week (see the `Instructions` tab in that file for what each column means). We drop the pre-filled example row and any blank rows automatically, so you don't have to remember to delete the example before running this.

In [ ]:
upcoming = pd.read_excel("../upcoming_week_template.xlsx", sheet_name="Upcoming_Week")

upcoming = upcoming[upcoming["date"].notna()]
upcoming = upcoming[~upcoming["notes"].astype(str).str.contains("EXAMPLE", case=False, na=False)]
upcoming = upcoming.reset_index(drop=True)

upcoming["date"] = pd.to_datetime(upcoming["date"])

if upcoming.empty:
    raise ValueError(
        "No real rows found in Upcoming_Week — fill in next week's dates in "
        "upcoming_week_template.xlsx (past the example row) before running this notebook."
    )

print(f"Loaded {len(upcoming)} upcoming date(s): {upcoming['date'].min().date()} to {upcoming['date'].max().date()}")
upcoming

**A safety check worth having:** confirm every upcoming date is within 7 days of the last historical date — if it isn't, `covers_lag_7` would need a covers value that doesn't exist yet, which is exactly the leakage risk Step 0 described.

In [ ]:
last_hist_date = hist["date"].max()
max_allowed = last_hist_date + pd.Timedelta(days=7)

too_far = upcoming[upcoming["date"] > max_allowed]
if len(too_far):
    raise ValueError(
        f"These dates are more than 7 days past the latest history ({last_hist_date.date()}) "
        f"and can't be forecast yet:\n{too_far['date'].dt.date.tolist()}\n"
        f"Add more recent actuals to covers_daily_clean.csv, or forecast a closer date."
    )
print("All upcoming dates are within the valid 7-day horizon.")

## Step 4: Build the same features for the upcoming dates

Same functions from Step 2, applied to the input sheet instead of history. `is_holiday`, `is_local_event`, `is_promo`, `temp_high_f`, `is_rain`, and `reservations_covers` all come straight from what you filled in; the lag and rolling-average features are looked up from real historical `covers`, which is why the Step 3 date check matters.

In [ ]:
upcoming["day_of_week"] = pd.Categorical(upcoming["date"].dt.day_name(), categories=dow_order)

upcoming["covers_lag_7"] = upcoming["date"].apply(lambda d: get_lag(d, 7))
upcoming["covers_lag_14"] = upcoming["date"].apply(lambda d: get_lag(d, 14))
upcoming["covers_rollavg_4wk_weekday"] = [
    get_rollavg_weekday(row.date, row.day_of_week, hist) for row in upcoming.itertuples()
]
upcoming["covers_rollavg_7d_trend"] = upcoming["date"].apply(get_rollavg_7d_trend)

upcoming = upcoming.rename(columns={
    "temp_high_f_forecast": "temp_high_f",
    "is_rain_forecast": "is_rain",
    "expected_reservations": "reservations_covers",
})

missing = upcoming[numeric_feats].isna().sum()
if missing.sum() > 0:
    print("Warning — some features are missing for the upcoming week:")
    print(missing[missing > 0])
else:
    print("All features present for every upcoming date.")

upcoming[["date", "day_of_week"] + numeric_feats]

## Step 5: Generate the forecast

In [ ]:
dow_upcoming = pd.get_dummies(upcoming["day_of_week"], prefix="dow", drop_first=True).astype(float)
dow_upcoming = dow_upcoming.reindex(columns=dow_dummies.columns, fill_value=0.0)  # match training columns exactly

X_upcoming = pd.concat([dow_upcoming, upcoming[numeric_feats]], axis=1)
X_upcoming_design = np.column_stack([np.ones(len(X_upcoming)), X_upcoming.values])

upcoming["forecast_covers"] = np.round(X_upcoming_design @ coefs).astype(int)

forecast_table = upcoming[[
    "date", "day_of_week", "forecast_covers",
    "is_holiday", "holiday_name", "is_local_event", "event_name",
    "is_promo", "promo_name", "reservations_covers", "temp_high_f", "is_rain", "notes",
]]
forecast_table

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#4C72B0" if w < 5 else "#C44E52" for w in upcoming["date"].dt.weekday]  # weekend bars highlighted
ax.bar(upcoming["date"].dt.strftime("%a %m/%d"), upcoming["forecast_covers"], color=colors)
ax.set_title("Next 7 days — forecast covers")
ax.set_ylabel("Forecast covers")
for i, v in enumerate(upcoming["forecast_covers"]):
    ax.text(i, v + 3, str(v), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

**A reminder on expected accuracy:** notebook 05's held-out test put this modeling approach at roughly 15-16 covers of typical error (MAE) — closer to 15 with reservations added in, as here. Treat each forecast as a solid planning number with that much give either way, not an exact count — and lean on the notes column and your own judgment for anything unusual the model has no way to see (private events, staff callouts, one-off closures).

## Step 6: Save the forecast for reference

In [ ]:
output_path = "../data/processed/latest_weekly_forecast.csv"
forecast_table.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

## Your weekly checklist

1. **Export last week's actuals** from POS/reservations (covers, reservations, any notes on closures or anomalies) and append them to `data/processed/covers_daily_clean.csv`, following the join/clean steps from notebook 01.
2. **Open `upcoming_week_template.xlsx`**, delete or overwrite the example row, and fill in the next 7 calendar dates: known holidays/events/promos, a weather forecast (temp + rain), and your best estimate of eventual reservations for each date.
3. **Run this notebook (06) top to bottom.** It automatically retrains on the refreshed history and reads your updated input sheet — no code changes needed week to week.
4. **Review the forecast table and chart.** Sanity-check against your own read of the week — does anything here contradict what you already know is happening?
5. **Use the forecast for staffing and prep**, per the ratios from notebook 05, Step 7 (e.g., ~1 server per 20 covers).
6. Repeat weekly. The model quietly gets better over time as more real weeks accumulate in the history file — no separate retraining process to remember.

## Recap

- Made `reservations_covers` a feature *when the data supports it* — the strongest single predictor available in this restaurant's history (0.94 correlation) — with an explicit caveat about entering a *projected final* number, not a partial current count. Restaurants that only track a covers backlog fall back automatically to the notebook 05 feature set, no code changes needed.
- Built reusable feature functions (`get_lag`, `get_rollavg_weekday`, `get_rollavg_7d_trend`) so the exact same logic applies to historical training rows and future forecast rows — no duplicated, possibly-inconsistent code between them.
- Capped the forecast horizon at 7 days, with an explicit check that refuses to forecast a date that would require an input we don't actually have yet.
- Read next week's known context from a manager-editable spreadsheet, generated a forecast, and saved it for reference — a full loop with no code changes required between weekly runs.

**Next up (the last step of the project):** a short written summary and a couple of portfolio-ready charts pulling together what this project found — useful both as your own reference and as something you could show in a hospitality/operations/analytics job application.